In [1]:
# Parameters
BATCH_MODE = "true"


# SC-11-LLM-Assisted - Developpement Smart Contracts Assiste par LLM

[<< Precedent : Account Abstraction](SC-10-Account-Abstraction.ipynb) | [Retour au sommaire](../README.md) | [Suivant : Foundry Testing >>](../03-Foundry-Testing/SC-12-Foundry-Testing.ipynb)

---

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Utiliser les **LLMs** (Claude, GPT) pour generer du code Solidity
2. Appliquer des **patterns de prompting** efficaces pour les smart contracts
3. Exploiter les LLMs pour la **detection de vulnerabilites**
4. Generer automatiquement la **documentation NatSpec**
5. Integrer l'API **Anthropic Claude** dans un workflow de developpement
6. Utiliser un **LLM local** (Qwen) comme alternative open-source

### Prerequis

- Notebooks SC-1 a SC-4 (fondements Solidity)
- Cle API OpenAI ou Anthropic (optionnel pour tests)
- Python 3.10+ avec `openai`, `anthropic`, `requests`

### Duree estimee : 45 minutes

---

## Introduction

Les **Large Language Models** ont revolutionne le developpement de smart contracts. Ils peuvent :

- **Generer du code** a partir de specifications en langage naturel
- **Auditer** des contrats pour detecter des vulnerabilites
- **Documenter** automatiquement avec des commentaires NatSpec
- **Expliquer** des concepts complexes Solidity

### Pourquoi utiliser les LLMs pour Solidity ?

| Avantage | Description |
|----------|-------------|
| **Rapidite** | Prototypage en minutes, pas en heures |
| **Apprentissage** | Explications contextuelles du code genere |
| **Securite** | Detection precoce de patterns vulnerables |
| **Documentation** | NatSpec genere automatiquement |

### Limitations importantes

> **Attention** : Les LLMs peuvent generer du code contenant des bugs ou vulnerabilites.
> Toujours auditer, tester et verifier formellement le code genere avant deploiement en production.

## 1. Configuration et Imports

In [2]:
# Imports standards
import os
import json
import re
from typing import Dict, List, Optional, Any
from dataclasses import dataclass
from pathlib import Path

# API clients - Optionnels avec graceful fallback
try:
    from openai import OpenAI
    HAS_OPENAI = True
except ImportError:
    HAS_OPENAI = False
    OpenAI = None

try:
    from anthropic import Anthropic
    HAS_ANTHROPIC = True
except ImportError:
    HAS_ANTHROPIC = False
    Anthropic = None

# Charger les variables d'environnement
try:
    from dotenv import load_dotenv
    load_dotenv()
    HAS_DOTENV = True
except ImportError:
    HAS_DOTENV = False

print("Imports charges avec succes")
print(f"OpenAI disponible: {HAS_OPENAI}")
print(f"Anthropic disponible: {HAS_ANTHROPIC}")

Imports charges avec succes
OpenAI disponible: True
Anthropic disponible: True


Configuration des clients API Anthropic et OpenAI avec gestion des clés d'authentification.

In [3]:
# Configuration des clients API
@dataclass
class LLMConfig:
    """Configuration pour les appels LLM"""
    openai_api_key: str = os.getenv("OPENAI_API_KEY", "") if HAS_DOTENV else ""
    anthropic_api_key: str = os.getenv("ANTHROPIC_API_KEY", "") if HAS_DOTENV else ""
    model_openai: str = "gpt-4o"
    model_anthropic: str = "claude-sonnet-4-6"
    temperature: float = 0.7
    max_tokens: int = 2000

config = LLMConfig()

# Initialiser les clients (si disponibles et configures)
openai_client = None
anthropic_client = None

if HAS_OPENAI and config.openai_api_key:
    try:
        openai_client = OpenAI(api_key=config.openai_api_key)
        print("Client OpenAI initialise")
    except Exception as e:
        print(f"Erreur initialisation OpenAI: {e}")

if HAS_ANTHROPIC and config.anthropic_api_key:
    try:
        anthropic_client = Anthropic(api_key=config.anthropic_api_key)
        print("Client Anthropic initialise")
    except Exception as e:
        print(f"Erreur initialisation Anthropic: {e}")

print(f"\nStatut:")
print(f"  OpenAI client: {'Configure' if openai_client else 'Non configure'}")
print(f"  Anthropic client: {'Configure' if anthropic_client else 'Non configure'}")


Statut:
  OpenAI client: Non configure
  Anthropic client: Non configure


### Interpretation

La configuration utilise des variables d'environnement pour les cles API. Le notebook fonctionne meme sans API configuree, en utilisant des exemples de mock responses.

**Pour configurer vos cles** :
1. Creez un fichier `.env` dans le repertoire
2. Ajoutez : `OPENAI_API_KEY=sk-...` ou `ANTHROPIC_API_KEY=sk-ant-...`

## 2. Prompt Patterns pour Solidity

Le prompting efficace pour Solidity necessite structure et precision. Voici les templates de base.

In [4]:
# Templates de prompts pour Solidity

SOLIDITY_PROMPTS = {
    "contract_generation": """Tu es un expert en developpement de smart contracts Solidity.

Genere un contrat Solidity complet pour le cas d'usage suivant:
{description}

Contraintes:
- Version Solidity: ^0.8.20
- Inclure les imports OpenZeppelin si necessaire
- Ajouter les commentaires NatSpec complets
- Implementer les checks de securite (ReentrancyGuard, etc.)
- Utiliser les patterns les plus recents et securises

Fournit uniquement le code Solidity, sans explications supplementaires.
""",

    "audit": """Tu es un auditeur de securite smart contracts expert.

Analyse ce contrat Solidity pour les vulnerabilites potentielles:
```solidity
{code}
```

Identifie:
1. Vulnerabilites connues (reentrancy, overflow, etc.)
2. Problems de gas optimization
3. Centralisation risks
4. Best practices non respectees

Format de sortie:
- SEVERITE (CRITICAL/HIGH/MEDIUM/LOW)
- Description du probleme
- Ligne concernee
- Recommandation de correction
""",

    "natspec": """Genere la documentation NatSpec complete pour ce contrat Solidity:
```solidity
{code}
```

Inclus pour chaque fonction:
- @title et @description pour le contrat
- @notice pour l'utilisateur
- @dev pour les details techniques
- @param pour chaque parametre
- @return pour les valeurs de retour

Retourne le code avec les commentaires NatSpec ajoutes.
""",

    "explain": """Explique ce code Solidity de maniere pedagogique:
```solidity
{code}
```

Structure ton explication:
1. Objectif global du contrat
2. Variables d'etat et leur role
3. Fonctions principales
4. Patterns de securite utilises
5. Cas d'usage typiques
"""
}

print("Templates de prompts definis:")
for name in SOLIDITY_PROMPTS:
    print(f"  - {name}")

Templates de prompts definis:
  - contract_generation
  - audit
  - natspec
  - explain


### 2.1 Bonnes pratiques de prompting

| Pratique | Exemple | Impact |
|----------|---------|--------|
| **Specifier la version** | "Solidity ^0.8.20" | Code compatible |
| **Inclure OpenZeppelin** | "Utilise IERC20 de OpenZeppelin" | Standards respectes |
| **Demander NatSpec** | "Ajoute les commentaires NatSpec" | Documentation auto |
| **Contraintes de securite** | "Utilise ReentrancyGuard" | Code plus sur |
| **Format de sortie** | "Retourne uniquement le code" | Parsing facilite |

## 3. Generation de Contrats via API

Implementons un generateur de smart contracts utilisant l'API Anthropic Claude.

In [5]:
def extract_solidity_code(response_text: str) -> str:
    """Extrait le code Solidite d'une reponse LLM"""
    # Chercher les blocs de code Solidity
    pattern = r'```solidity\s*\n(.*?)```'
    matches = re.findall(pattern, response_text, re.DOTALL)
    
    if matches:
        return matches[0].strip()
    
    # Fallback: chercher n'importe quel bloc de code
    pattern = r'```\s*\n(.*?)```'
    matches = re.findall(pattern, response_text, re.DOTALL)
    
    if matches:
        return matches[0].strip()
    
    return response_text


def generate_contract(
    description: str,
    client_type: str = "anthropic",
    temperature: float = 0.7
) -> tuple:
    """
    Genere un contrat Solidity via LLM.
    Retourne (code_solide, reponse_complete)
    """
    prompt = SOLIDITY_PROMPTS["contract_generation"].format(description=description)
    
    if client_type == "anthropic" and anthropic_client:
        try:
            response = anthropic_client.messages.create(
                model=config.model_anthropic,
                max_tokens=config.max_tokens,
                temperature=temperature,
                messages=[{"role": "user", "content": prompt}]
            )
            full_response = response.content[0].text
            solidity_code = extract_solidity_code(full_response)
            return solidity_code, full_response
        except Exception as e:
            return None, f"Erreur API Anthropic: {e}"
    
    elif client_type == "openai" and openai_client:
        try:
            response = openai_client.chat.completions.create(
                model=config.model_openai,
                max_tokens=config.max_tokens,
                temperature=temperature,
                messages=[{"role": "user", "content": prompt}]
            )
            full_response = response.choices[0].message.content
            solidity_code = extract_solidity_code(full_response)
            return solidity_code, full_response
        except Exception as e:
            return None, f"Erreur API OpenAI: {e}"
    
    else:
        # Mock response quand pas d'API
        mock_response = f"""```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/// @title Contrat genere par LLM (mock)
/// @dev Ceci est un exemple de mock - configurez l'API pour du vrai code
contract GeneratedContract {{
    // Implementation basee sur: {description[:50]}...
    
    address public owner;
    
    constructor() {{
        owner = msg.sender;
    }}
    
    modifier onlyOwner() {{
        require(msg.sender == owner, "Not owner");
        _;
    }}
}}
```"""
        return extract_solidity_code(mock_response), mock_response


print("Fonction generate_contract definie")

Fonction generate_contract definie


Génération d'un contrat Solidity simple à partir d'une description en langage naturel via l'API LLM.

In [6]:
# Exemple: Generer un contrat simple
description = """
Un contrat de stockage simple qui permet:
- Stocker une valeur uint256
- Recuperer la valeur stockee
- Seul le owner peut modifier la valeur
- Emettre un event quand la valeur change
"""

print("Generation d'un contrat de stockage...")
print(f"Description: {description.strip()}")
print("\n" + "="*60)

code, response = generate_contract(description)

if code:
    print("CONTRAT GENERE:")
    print("="*60)
    print(code)
else:
    print(f"Erreur: {response}")

Generation d'un contrat de stockage...
Description: Un contrat de stockage simple qui permet:
- Stocker une valeur uint256
- Recuperer la valeur stockee
- Seul le owner peut modifier la valeur
- Emettre un event quand la valeur change

CONTRAT GENERE:
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/// @title Contrat genere par LLM (mock)
/// @dev Ceci est un exemple de mock - configurez l'API pour du vrai code
contract GeneratedContract {
    // Implementation basee sur: 
Un contrat de stockage simple qui permet:
- Stock...
    
    address public owner;
    
    constructor() {
        owner = msg.sender;
    }
    
    modifier onlyOwner() {
        require(msg.sender == owner, "Not owner");
        _;
    }
}


### Interpretation : Generation de contrat par LLM

Le contrat genere (mode mock) illustre les elements structurels minimaux que tout LLM devrait produire.

| Element | Present dans le mock | Role |
|---------|---------------------|------|
| **SPDX License** | Oui (`MIT`) | Obligatoire pour la compilation |
| **Pragma** | Oui (`^0.8.20`) | Fixe la version du compilateur |
| **NatSpec** | Partiel (mock) | Documentation pour Etherscan et les IDEs |
| **Access control** | Oui (`onlyOwner`) | Protection des fonctions sensibles |
| **Event** | Non (manquant dans le mock) | Tracking on-chain des changements d'etat |

**Ecart entre description et resultat mock** : la description demandait un event `ValueChanged`, une fonction de stockage et une de lecture. Le mock retourne un template generique sans ces elements. Avec une API configuree (Anthropic ou OpenAI), le LLM adapterait le code exactement a la description, incluant l'event et les deux fonctions.

### Interpretation

Le contrat genere devrait inclure :

| Element | Description |
|---------|-------------|
| **SPDX License** | Identifiant de licence MIT |
| **Pragma** | Version Solidity ciblee |
| **NatSpec** | Documentation des fonctions |
| **Event** | ValueChanged pour tracking |
| **Modifier** | onlyOwner pour access control |

## 4. Audit de Securite avec LLM

Utilisons les LLMs pour detecter les vulnerabilites dans les smart contracts.

In [7]:
def audit_contract(code: str, client_type: str = "anthropic") -> str:
    """
    Audite un contrat Solidity pour les vulnerabilites.
    """
    prompt = SOLIDITY_PROMPTS["audit"].format(code=code)
    
    if client_type == "anthropic" and anthropic_client:
        try:
            response = anthropic_client.messages.create(
                model=config.model_anthropic,
                max_tokens=config.max_tokens,
                messages=[{"role": "user", "content": prompt}]
            )
            return response.content[0].text
        except Exception as e:
            return f"Erreur API: {e}"
    
    elif client_type == "openai" and openai_client:
        try:
            response = openai_client.chat.completions.create(
                model=config.model_openai,
                max_tokens=config.max_tokens,
                messages=[{"role": "user", "content": prompt}]
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Erreur API: {e}"
    
    else:
        # Mock audit response
        return """## Audit de Securite (Mock Response)

### VULNERABILITES DETECTEES:

**CRITICAL - Reentrancy Risk**
- Ligne: N/A
- Description: Ce mock ne peut pas analyser le code reel
- Recommandation: Configurez l'API pour un audit reel

**HIGH - Centralisation Risk**
- Ligne: constructor
- Description: Le pattern owner unique cree un point de centralisation
- Recommandation: Considerer un systeme multi-sig pour la production

### RECOMMANDATIONS:
1. Utiliser ReentrancyGuard de OpenZeppelin
2. Ajouter des tests avec Foundry
3. Effectuer une verification formelle avec Certora
"""


print("Fonction audit_contract definie")

Fonction audit_contract definie


Exemple de génération NatSpec : ajout automatique de documentation à un contrat sans commentaires.

In [8]:
# Exemple: Auditer un contrat vulnerables
vulnerable_contract = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.0;

contract VulnerableVault {
    mapping(address => uint256) public balances;
    
    function deposit() external payable {
        balances[msg.sender] += msg.value;
    }
    
    function withdraw() external {
        uint256 amount = balances[msg.sender];
        // VULNERABILITE: appel externe avant mise a jour
        (bool success, ) = msg.sender.call{value: amount}("");
        require(success, "Transfer failed");
        balances[msg.sender] = 0;
    }
}
"""

print("=== AUDIT DE SECURITE ===")
print("Contrat analyse: VulnerableVault")
print("\n" + "="*60)

audit_result = audit_contract(vulnerable_contract)
print(audit_result)

=== AUDIT DE SECURITE ===
Contrat analyse: VulnerableVault

## Audit de Securite (Mock Response)

### VULNERABILITES DETECTEES:

**CRITICAL - Reentrancy Risk**
- Ligne: N/A
- Description: Ce mock ne peut pas analyser le code reel
- Recommandation: Configurez l'API pour un audit reel

**HIGH - Centralisation Risk**
- Ligne: constructor
- Description: Le pattern owner unique cree un point de centralisation
- Recommandation: Considerer un systeme multi-sig pour la production

### RECOMMANDATIONS:
1. Utiliser ReentrancyGuard de OpenZeppelin
2. Ajouter des tests avec Foundry
3. Effectuer une verification formelle avec Certora



### Interpretation : Audit de VulnerableVault

Le contrat `VulnerableVault` contient une **reentrancy classique**, la vulnerabilite la plus citee dans les hacks DeFi.

**Sequence d'attaque** :

1. L'attaquant depose 1 ETH via `deposit()`
2. Il appelle `withdraw()` — le contrat envoie 1 ETH via `msg.sender.call{value: amount}("")`
3. Le fallback de l'attaquant **reappelle** `withdraw()` avant que `balances[msg.sender] = 0` ne s'execute
4. Le solde est encore a 1 ETH → retrait d'un second ETH
5. Repetition jusqu'a drainage complet du contrat

**Pattern correctif (Checks-Effects-Interactions)** :

```solidity
function withdraw() external {
    uint256 amount = balances[msg.sender];  // Checks
    balances[msg.sender] = 0;               // Effects (AVANT l'appel externe)
    (bool success, ) = msg.sender.call{value: amount}("");  // Interactions
    require(success, "Transfer failed");
}
```

La regle est simple : toute mise a jour de l'etat (**Effects**) doit preceder tout appel externe (**Interactions**). La variable `amount` est lue avant la mise a zero (**Checks**), garantissant que le reappel trouve un solde a zero.

### Interpretation

L'audit LLM devrait identifier la **reentrancy vulnerability** dans le contrat ci-dessus :

```
1. Utilisateur A appelle withdraw()
2. Le contrat envoie les fonds a A
3. A est un contrat malveillant qui reappelle withdraw()
4. Le solde n'a pas encore ete mis a jour (toujours > 0)
5. Le contrat envoie les fonds une deuxieme fois !
```

**Correction recommandee** : Pattern Checks-Effects-Interactions ou ReentrancyGuard.

## 5. Generation de Documentation NatSpec

La documentation NatSpec (Ethereum Natural Language Specification) est essentielle pour les smart contracts.

In [9]:
def generate_natspec(code: str, client_type: str = "anthropic") -> str:
    """
    Genere la documentation NatSpec pour un contrat.
    """
    prompt = SOLIDITY_PROMPTS["natspec"].format(code=code)
    
    if client_type == "anthropic" and anthropic_client:
        try:
            response = anthropic_client.messages.create(
                model=config.model_anthropic,
                max_tokens=config.max_tokens,
                messages=[{"role": "user", "content": prompt}]
            )
            return response.content[0].text
        except Exception as e:
            return f"Erreur API: {e}"
    
    elif client_type == "openai" and openai_client:
        try:
            response = openai_client.chat.completions.create(
                model=config.model_openai,
                max_tokens=config.max_tokens,
                messages=[{"role": "user", "content": prompt}]
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Erreur API: {e}"
    
    else:
        # Mock NatSpec generation
        return f"""```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/// @title SimpleStorage
/// @author LLM Generated
/// @notice Un contrat simple pour stocker et recuperer une valeur
/// @dev Utilise le pattern Ownable pour le controle d'acces
contract SimpleStorage {{
    /// @notice La valeur stockee dans le contrat
    /// @dev Visible publiquement, modifiable uniquement par le owner
    uint256 public storedValue;
    
    /// @notice Adresse du proprietaire du contrat
    address public owner;
    
    /// @notice Emet quand la valeur est modifiee
    /// @param oldValue L'ancienne valeur
    /// @param newValue La nouvelle valeur
    /// @param changedBy L'adresse qui a effectue le changement
    event ValueChanged(uint256 oldValue, uint256 newValue, address changedBy);
    
    /// @notice Initialise le contrat avec le sender comme owner
    constructor() {{
        owner = msg.sender;
    }}
    
    /// @notice Modifie la valeur stockee
    /// @dev Seul le owner peut appeler cette fonction
    /// @param newValue La nouvelle valeur a stocker
    function setValue(uint256 newValue) external {{
        require(msg.sender == owner, "Only owner can set value");
        uint256 oldValue = storedValue;
        storedValue = newValue;
        emit ValueChanged(oldValue, newValue, msg.sender);
    }}
    
    /// @notice Recupere la valeur stockee
    /// @return La valeur actuellement stockee
    function getValue() external view returns (uint256) {{
        return storedValue;
    }}
}}
```"""


print("Fonction generate_natspec definie")

Fonction generate_natspec definie


Exemple d'audit automatique d'un contrat volontairement vulnérable pour identifier les failles de sécurité.

In [10]:
# Exemple: Ajouter NatSpec a un contrat sans documentation
undocumented_contract = """
pragma solidity ^0.8.20;

contract Token {
    string public name = "MyToken";
    string public symbol = "MTK";
    uint8 public decimals = 18;
    uint256 public totalSupply;
    mapping(address => uint256) public balanceOf;
    
    event Transfer(address indexed from, address indexed to, uint256 value);
    
    constructor(uint256 _initialSupply) {
        totalSupply = _initialSupply * 10 ** uint256(decimals);
        balanceOf[msg.sender] = totalSupply;
    }
    
    function transfer(address _to, uint256 _value) public returns (bool success) {
        require(balanceOf[msg.sender] >= _value, "Insufficient balance");
        balanceOf[msg.sender] -= _value;
        balanceOf[_to] += _value;
        emit Transfer(msg.sender, _to, _value);
        return true;
    }
}
"""

print("=== GENERATION NATSPEC ===")
print("Contrat original sans documentation")
print("\n" + "="*60)

documented = generate_natspec(undocumented_contract)
print(documented)

=== GENERATION NATSPEC ===
Contrat original sans documentation

```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/// @title SimpleStorage
/// @author LLM Generated
/// @notice Un contrat simple pour stocker et recuperer une valeur
/// @dev Utilise le pattern Ownable pour le controle d'acces
contract SimpleStorage {
    /// @notice La valeur stockee dans le contrat
    /// @dev Visible publiquement, modifiable uniquement par le owner
    uint256 public storedValue;
    
    /// @notice Adresse du proprietaire du contrat
    address public owner;
    
    /// @notice Emet quand la valeur est modifiee
    /// @param oldValue L'ancienne valeur
    /// @param newValue La nouvelle valeur
    /// @param changedBy L'adresse qui a effectue le changement
    event ValueChanged(uint256 oldValue, uint256 newValue, address changedBy);
    
    /// @notice Initialise le contrat avec le sender comme owner
    constructor() {
        owner = msg.sender;
    }
    
    /// @notice 

### Interpretation

La documentation NatSpec generee inclut les tags standards :

| Tag | Usage |
|-----|-------|
| `@title` | Nom du contrat |
| `@author` | Auteur du code |
| `@notice` | Description utilisateur |
| `@dev` | Details techniques |
| `@param` | Description des parametres |
| `@return` | Description de la valeur de retour |
| `@event` | Description des evenements |

## 6. Integration avec l'API Anthropic Claude

Exemple d'integration complete avec Claude Sonnet 4.6 pour un workflow de developpement.

In [11]:
class SolidityLLMAssistant:
    """
    Assistant LLM pour le developpement de smart contracts.
    """
    
    def __init__(self, client_type: str = "anthropic"):
        self.client_type = client_type
        self.client = anthropic_client if client_type == "anthropic" else openai_client
        self.model = config.model_anthropic if client_type == "anthropic" else config.model_openai
    
    def generate(
        self,
        description: str,
        temperature: float = 0.7,
        max_tokens: int = 2000
    ) -> Dict[str, Any]:
        """Genere un contrat complet avec metadonnees"""
        code, response = generate_contract(description, self.client_type)
        
        return {
            "code": code,
            "full_response": response,
            "model": self.model,
            "provider": self.client_type,
            "success": code is not None
        }
    
    def audit(self, code: str) -> Dict[str, Any]:
        """Audite un contrat pour les vulnerabilites"""
        report = audit_contract(code, self.client_type)
        
        return {
            "report": report,
            "model": self.model,
            "provider": self.client_type
        }
    
    def document(self, code: str) -> Dict[str, Any]:
        """Ajoute la documentation NatSpec"""
        documented = generate_natspec(code, self.client_type)
        
        return {
            "documented_code": extract_solidity_code(documented),
            "full_response": documented,
            "model": self.model,
            "provider": self.client_type
        }
    
    def full_workflow(self, description: str) -> Dict[str, Any]:
        """
        Workflow complet: generation -> audit -> documentation
        """
        # 1. Generer
        gen_result = self.generate(description)
        if not gen_result["success"]:
            return {"error": "Generation failed", "details": gen_result}
        
        # 2. Auditer
        audit_result = self.audit(gen_result["code"])
        
        # 3. Documenter
        doc_result = self.document(gen_result["code"])
        
        return {
            "original_code": gen_result["code"],
            "documented_code": doc_result["documented_code"],
            "audit_report": audit_result["report"],
            "model": self.model,
            "provider": self.client_type
        }


# Initialiser l'assistant
assistant = SolidityLLMAssistant(client_type="anthropic" if anthropic_client else "openai")
print(f"Assistant initialise avec {assistant.client_type} ({assistant.model})")

Assistant initialise avec openai (gpt-4o)


Démonstration du workflow complet : génération, audit et documentation d'un contrat Solidity via l'assistant LLM.

In [12]:
# Demo du workflow complet
print("=== WORKFLOW COMPLET LLM ===")
print("\nDescription: Un compteur avec increment et decrement")
print("\n" + "="*60)

result = assistant.full_workflow("""
Un contrat Counter qui permet:
- Incrementer un compteur
- Decrementer le compteur (minimum 0)
- Lire la valeur actuelle
- Seul le owner peut incrementer/decrementer
""")

if "error" not in result:
    print("CODE GENERE:")
    print("-" * 40)
    print(result["original_code"][:500] + "..." if len(result["original_code"] or "") > 500 else result["original_code"])
    
    print("\n" + "="*60)
    print("AUDIT (extrait):")
    print("-" * 40)
    print(result["audit_report"][:400] + "..." if len(result["audit_report"] or "") > 400 else result["audit_report"])
else:
    print(f"Erreur: {result['error']}")

=== WORKFLOW COMPLET LLM ===

Description: Un compteur avec increment et decrement

CODE GENERE:
----------------------------------------
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/// @title Contrat genere par LLM (mock)
/// @dev Ceci est un exemple de mock - configurez l'API pour du vrai code
contract GeneratedContract {
    // Implementation basee sur: 
Un contrat Counter qui permet:
- Incrementer un c...
    
    address public owner;
    
    constructor() {
        owner = msg.sender;
    }
    
    modifier onlyOwner() {
        require(msg.sender == owner, "Not owner");
        _;
    }
}

AUDIT (extrait):
----------------------------------------
## Audit de Securite (Mock Response)

### VULNERABILITES DETECTEES:

**CRITICAL - Reentrancy Risk**
- Ligne: N/A
- Description: Ce mock ne peut pas analyser le code reel
- Recommandation: Configurez l'API pour un audit reel

**HIGH - Centralisation Risk**
- Ligne: constructor
- Description: Le pattern owner unique cree u

### Interpretation : Workflow complet LLM

Le workflow en trois etapes illustre le pipeline industriel de generation assistee.

| Etape | Entree | Sortie | Point d'attention |
|-------|--------|--------|-------------------|
| **Generation** | Description NL | Code Solidity | Le mock retourne un template generique ; un vrai LLM adapterait le code a la description |
| **Audit** | Code genere | Rapport de vulnerabilites | Le mock signale des risques generiques ; un vrai audit identifierait la reentrancy specifique |
| **Documentation** | Code audite | Code + NatSpec | Ajoute les tags @notice, @dev, @param, @return |

**Limites du mode mock** : sans API configuree, les trois etapes retournent des reponses generiques. Le contrat genere ne correspond pas a la description (il manque le compteur, l'increment, le decrement), et l'audit ne detecte pas les vraies vulnerabilites du code. Configurer une cle API (`ANTHROPIC_API_KEY` ou `OPENAI_API_KEY` dans `.env`) active les veritables appels LLM et produit un code specifique au prompt.

## 7. LLM Local avec Qwen (Optionnel)

Pour les cas ou les API cloud ne sont pas disponibles ou souhaites, on peut utiliser un LLM local via un endpoint.

In [13]:
try:
    import requests
except ImportError:
    print("Installation requise : pip install requests")
    raise

class LocalLLMAssistant:
    """
    Assistant utilisant un LLM local (ex: Qwen via vLLM ou Ollama).
    """
    
    def __init__(
        self,
        endpoint: str = "http://localhost:11434/api/generate",
        model: str = "qwen2.5-coder:7b"
    ):
        self.endpoint = endpoint
        self.model = model
        self.available = self._check_availability()
    
    def _check_availability(self) -> bool:
        """Verifie si le serveur local est disponible"""
        try:
            response = requests.get(
                self.endpoint.replace("/api/generate", ""),
                timeout=2
            )
            return response.status_code == 200
        except:
            return False
    
    def generate(self, prompt: str, temperature: float = 0.7) -> str:
        """Genere une reponse via le LLM local"""
        if not self.available:
            return "LLM local non disponible. Demarrez Ollama ou vLLM."
        
        try:
            # Format Ollama
            response = requests.post(
                self.endpoint,
                json={
                    "model": self.model,
                    "prompt": prompt,
                    "temperature": temperature,
                    "stream": False
                },
                timeout=60
            )
            
            if response.status_code == 200:
                return response.json().get("response", "")
            return f"Erreur: {response.status_code}"
        
        except Exception as e:
            return f"Erreur de connexion: {e}"
    
    def generate_contract(self, description: str) -> str:
        """Genere un contrat Solidity"""
        prompt = SOLIDITY_PROMPTS["contract_generation"].format(description=description)
        response = self.generate(prompt)
        return extract_solidity_code(response)


# Initialiser l'assistant local
local_assistant = LocalLLMAssistant()
print(f"LLM local disponible: {local_assistant.available}")
print(f"Endpoint: {local_assistant.endpoint}")
print(f"Modele: {local_assistant.model}")

LLM local disponible: False
Endpoint: http://localhost:11434/api/generate
Modele: qwen2.5-coder:7b


Démonstration de l'assistant LLM local avec un modèle hébergé en local (si disponible) pour la génération de contrats.

In [14]:
# Demo LLM local (si disponible)
print("=== TEST LLM LOCAL ===")

if local_assistant.available:
    code = local_assistant.generate_contract("Un simple contrat HelloWorld")
    print("Contrat genere localement:")
    print(code)
else:
    print("LLM local non disponible.")
    print("\nPour utiliser un LLM local:")
    print("  1. Installer Ollama: https://ollama.ai")
    print("  2. Telecharger le modele: ollama pull qwen2.5-coder:7b")
    print("  3. Demarrer Ollama: ollama serve")

=== TEST LLM LOCAL ===
LLM local non disponible.

Pour utiliser un LLM local:
  1. Installer Ollama: https://ollama.ai
  2. Telecharger le modele: ollama pull qwen2.5-coder:7b
  3. Demarrer Ollama: ollama serve


### Comparaison API Cloud vs Local

| Aspect | API Cloud (Claude/GPT) | LLM Local (Qwen) |
|--------|----------------------|------------------|
| **Qualite code** | Excellente | Bonne |
| **Latence** | 1-5 secondes | 5-30 secondes |
| **Cout** | Par token | Gratuit |
| **Confidentialite** | Donnees envoyees | Tout local |
| **Disponibilite** | 99.9% | Depend du hardware |
| **Setup** | Cle API | Installation locale |

## 8. Resume et Bonnes Pratiques

### Ce que nous avons appris

1. **Prompting pour Solidity** : Structure et precision sont essentielles
2. **Generation de contrats** : LLMs peuvent produire du code fonctionnel rapidement
3. **Audit de securite** : Detection de vulnerabilites assistee par IA
4. **Documentation NatSpec** : Generation automatique des commentaires
5. **API Claude** : Integration avec Anthropic pour des resultats optimaux
6. **LLM Local** : Alternative open-source avec Qwen/Ollama

### Checklist avant deploiement

- [ ] Code genere par LLM audite manuellement
- [ ] Tests unitaires ecrits avec Foundry
- [ ] Fuzz testing effectue
- [ ] Audit formel avec Certora (si critique)
- [ ] Documentation NatSpec complete
- [ ] Code review par un expert humain

## Resume et perspectives

Ce notebook a explore l'integration des Large Language Models dans le workflow de developpement de smart contracts. Nous avons defini quatre templates de prompting specialises (generation de contrats, audit de securite, documentation NatSpec, explication pedagogique) et demontre leur utilisation via les API Anthropic Claude et OpenAI. L'audit assiste par LLM a ete illustre avec le cas classique de la reentrancy dans `VulnerableVault`, ou le pattern Checks-Effects-Interactions constitue la correction canonique.

Le workflow complet (generation, audit, documentation) implemente dans la classe `SolidityLLMAssistant` montre comment automatiser le pipeline industriel. L'alternative locale avec Qwen via Ollama offre une option sans cout et confidentielle, bien que la qualite du code genere soit inferieure aux modeles cloud. La limitation fondamentale demeure : le code genere par LLM doit toujours etre audite manuellement, teste avec Foundry et verifie formellement avant tout deploiement en production.

Le notebook suivant, [SC-12-Foundry-Testing](../03-Foundry-Testing/SC-12-Foundry-Testing.ipynb), introduit le framework Foundry pour ecrire des tests rigoureux en Solidite pur, etape indispensable pour valider tout code -- y compris celui genere par IA.

## Exemples guidés et exercices

Les deux **exemples guidés** ci-dessous sont des resolutions d'etudiants des promotions precedentes, conservees comme demonstrations. Chaque exemple est suivi d'un **nouvel exercice** a faire vous-meme. Les exercices 3 et 4 (plus bas) restent a completer.

### Exemple guidé 1 : Generer un Token ERC-20

> Contribution etudiante : **@Tooom123 (tom)** — PR #2474. Resolution conservee comme exemple guide.

Utilisation de l'assistant LLM pour generer un token ERC-20 complet (nom, symbole, decimales, fonctions mint/burn owner-only, events `Transfer`/`Approval`, documentation NatSpec), puis audit du code genere.

In [15]:
# Exemple guide 1 - Generer un ERC-20 avec l'assistant LLM (contribution @Tooom123)
# Etape: Generer un ERC-20 avec l'assistant LLM
# Etape: Auditer le code genere

erc20_description = """
Un token ERC-20 avec les caracteristiques suivantes:
- Nom: MyToken, symbole: MTK, 18 decimales
- Supply initiale de 1 000 000 tokens alloues au deploiement
- Fonctions mint et burn reservees au owner
- Events Transfer et Approval
- Documentation NatSpec complete
"""

result = assistant.full_workflow(erc20_description)

print("Code genere:")
print(result["original_code"])
print("\nRapport d'audit:")
print(result["audit_report"])

Code genere:
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/// @title Contrat genere par LLM (mock)
/// @dev Ceci est un exemple de mock - configurez l'API pour du vrai code
contract GeneratedContract {
    // Implementation basee sur: 
Un token ERC-20 avec les caracteristiques suivant...
    
    address public owner;
    
    constructor() {
        owner = msg.sender;
    }
    
    modifier onlyOwner() {
        require(msg.sender == owner, "Not owner");
        _;
    }
}

Rapport d'audit:
## Audit de Securite (Mock Response)

### VULNERABILITES DETECTEES:

**CRITICAL - Reentrancy Risk**
- Ligne: N/A
- Description: Ce mock ne peut pas analyser le code reel
- Recommandation: Configurez l'API pour un audit reel

**HIGH - Centralisation Risk**
- Ligne: constructor
- Description: Le pattern owner unique cree un point de centralisation
- Recommandation: Considerer un systeme multi-sig pour la production

### RECOMMANDATIONS:
1. Utiliser ReentrancyGuard de OpenZeppelin
2. Aj

### Exercice 1 (a vous) : Generer un contrat de sequestre (Escrow)

En vous inspirant de l'exemple guide ci-dessus (meme structure, description differente), utilisez `assistant.full_workflow()` pour generer un contrat de **sequestre (escrow)** a deux parties :
- un acheteur depose des fonds,
- le vendeur ne peut les recuperer qu'apres confirmation de l'acheteur,
- l'acheteur peut se faire rembourser si la livraison n'a pas lieu.

Affichez le code genere puis le rapport d'audit.

In [16]:
# Exercice 1 (a vous) - Espace de travail
# Etape 1 : Decrire un contrat de sequestre (escrow) a deux parties (acheteur / vendeur)
# Etape 2 : Appeler assistant.full_workflow(escrow_description)
# Etape 3 : Afficher result["original_code"] puis result["audit_report"]
# Indice : reprenez la structure de l'exemple guide 1 ci-dessus
escrow_description = """
# Votre description du contrat de sequestre (escrow) ici
"""
# result = assistant.full_workflow(escrow_description)
# print(result["original_code"])
# print(result["audit_report"])
print("Exercice a completer")

Exercice a completer


### Exemple guidé 2 : Workflow de Securite (boucle de generation auto-corrigee)

> Contribution etudiante : **@Tooom123 (tom)** — PR #2474. Resolution conservee comme exemple guide.

Une fonction qui : (1) genere un contrat a partir d'une description, (2) audite le code, (3) si des vulnerabilites CRITICAL/HIGH sont detectees, demande au LLM de corriger, puis (4) recommence jusqu'a un audit propre ou `max_iterations`. Testee ici sur un contrat de vote simple.

In [17]:
# Exemple guide 2 - Boucle de generation securisee (contribution @Tooom123)
def secure_generation_loop(description: str, max_iterations: int = 3):
    """
    Boucle de generation securisee avec auto-correction.

    1. Generer le contrat
    2. Auditer
    3. Si vulnerabilites -> corriger et recommencer
    4. Retourner le contrat final et le rapport d'audit
    """
    code, _ = generate_contract(description)
    audit = audit_contract(code)

    for i in range(max_iterations - 1):
        mots_critiques = ["CRITICAL", "HIGH"]
        vulnerabilite_trouvee = any(mot in audit for mot in mots_critiques)

        if not vulnerabilite_trouvee:
            break

        correction_prompt = f"""Le contrat suivant a des vulnerabilites detectees par un audit:

Contrat:
{code}

Rapport d'audit:
{audit}

Corrige toutes les vulnerabilites CRITICAL et HIGH et retourne le contrat Solidity corrige."""

        if anthropic_client:
            response = anthropic_client.messages.create(
                model=config.model_anthropic,
                max_tokens=config.max_tokens,
                messages=[{"role": "user", "content": correction_prompt}]
            )
            code = extract_solidity_code(response.content[0].text)
        elif openai_client:
            response = openai_client.chat.completions.create(
                model=config.model_openai,
                max_tokens=config.max_tokens,
                messages=[{"role": "user", "content": correction_prompt}]
            )
            code = extract_solidity_code(response.choices[0].message.content)
        else:
            # Pas de client LLM configure (mode mock) : on ne peut pas corriger, on s'arrete
            break

        audit = audit_contract(code)

    return {"code_final": code, "rapport_audit": audit}


voting_description = """
Un contrat de vote ou les utilisateurs peuvent voter pour des candidats.
Chaque adresse ne peut voter qu'une seule fois.
Le owner peut ajouter des candidats.
"""

resultat = secure_generation_loop(voting_description)
print("Code final:")
print(resultat["code_final"])
print("\nRapport d'audit final:")
print(resultat["rapport_audit"])

Code final:
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/// @title Contrat genere par LLM (mock)
/// @dev Ceci est un exemple de mock - configurez l'API pour du vrai code
contract GeneratedContract {
    // Implementation basee sur: 
Un contrat de vote ou les utilisateurs peuvent vo...
    
    address public owner;
    
    constructor() {
        owner = msg.sender;
    }
    
    modifier onlyOwner() {
        require(msg.sender == owner, "Not owner");
        _;
    }
}

Rapport d'audit final:
## Audit de Securite (Mock Response)

### VULNERABILITES DETECTEES:

**CRITICAL - Reentrancy Risk**
- Ligne: N/A
- Description: Ce mock ne peut pas analyser le code reel
- Recommandation: Configurez l'API pour un audit reel

**HIGH - Centralisation Risk**
- Ligne: constructor
- Description: Le pattern owner unique cree un point de centralisation
- Recommandation: Considerer un systeme multi-sig pour la production

### RECOMMANDATIONS:
1. Utiliser ReentrancyGuard de OpenZeppelin


### Exercice 2 (a vous) : Journaliser les iterations de la boucle securisee

A partir de l'exemple guide 2 ci-dessus, ecrivez une variante `secure_generation_loop_journalisee` qui, **a chaque iteration**, enregistre dans une liste le numero d'iteration et le nombre d'occurrences de `CRITICAL` et `HIGH` dans le rapport d'audit, puis retourne cet historique en plus du code final. Cela permet d'analyser la convergence de la boucle.

**Indice** : `audit.count("CRITICAL")`, `audit.count("HIGH")`.

In [18]:
# Exercice 2 (a vous) - Espace de travail
# Objectif : variante de secure_generation_loop qui journalise chaque iteration
# Etape 1 : Repartir de l'exemple guide 2 ci-dessus
# Etape 2 : A chaque iteration, ajouter {"iteration": i, "critical": n, "high": m} a une liste
# Etape 3 : Retourner {"code_final": ..., "rapport_audit": ..., "historique": [...]}
# Indice : audit.count("CRITICAL"), audit.count("HIGH")
def secure_generation_loop_journalisee(description: str, max_iterations: int = 3):
    historique = []  # liste de dicts {"iteration", "critical", "high"}
    # TODO etudiant : implementer la variante avec journal des iterations
    return None  # TODO : retourner code_final, rapport_audit et historique

# test_description = "Un contrat de tirage au sort (lottery) simple"
# print(secure_generation_loop_journalisee(test_description))
print("Exercice a completer")

Exercice a completer


### Exercice 3 : Generer la documentation NatSpec d'un contrat

Reutilisez `generate_natspec()` (section 5) pour documenter automatiquement un contrat
sans commentaires. Comptez les tags `@notice` dans la sortie.

**Indice** : `generate_natspec` retourne le code documente. Comptez `"@notice"` dedans.


In [19]:
# Exercice 3 - Generer la documentation NatSpec d'un contrat
# TODO etudiant : utiliser generate_natspec() pour documenter un contrat sans commentaires
# Etape 1 : Definir contract_a_documenter (un contrat Solidity minimal sans NatSpec)
# Etape 2 : Appeler generate_natspec(contract_a_documenter)
# Etape 3 : Extraire le code et compter les tags @notice
contract_a_documenter = """
pragma solidity ^0.8.20;

contract Counter {
    uint256 public count;
    function increment() public { count += 1; }
    function reset() public { count = 0; }
}
"""
# documented = generate_natspec(contract_a_documenter)
# code = extract_solidity_code(documented)
# print(code)
# print("Tags @notice trouves :", code.count("@notice"))
print("Exercice a completer")


Exercice a completer


## Ressources Complementaires

### Documentation
- [Anthropic API Docs](https://docs.anthropic.com/)
- [OpenAI API Docs](https://platform.openai.com/docs/)
- [Ollama](https://ollama.ai/) - LLM local
- [Solidity NatSpec](https://docs.soliditylang.org/en/latest/natspec-format.html)

### Outils
- [Cursor](https://cursor.sh/) - IDE avec LLM integre
- [GitHub Copilot](https://github.com/features/copilot) - Autocompletion IA
- [Slither](https://github.com/crytic/slither) - Analyse statique Solidity

---

[<< Precedent : Account Abstraction](SC-10-Account-Abstraction.ipynb) | [Retour au sommaire](../README.md) | [Suivant : Foundry Testing >>](../03-Foundry-Testing/SC-12-Foundry-Testing.ipynb)

## Conclusion

Ce notebook a explore l'integration des Large Language Models dans le workflow de developpement de smart contracts Solidity. Vous avez appris a utiliser quatre templates de prompting specialises (generation, audit de securite, documentation NatSpec, explication pedagogique), a implementer un pipeline complet de generation-audit-documentation via la classe `SolidityLLMAssistant`, et a utiliser un LLM local (Qwen via Ollama) comme alternative ouverte et confidentielle.

Les points cles a retenir sont la necessite de structurer les prompts avec des contraintes precises (version Solidity, OpenZeppelin, NatSpec), la detection de vulnerabilites classiques comme la reentrancy via l'audit assiste, et surtout la limitation fondamentale : tout code genere par LLM doit etre audite manuellement et teste avec Foundry avant deploiement en production.

### Exercice 4 : Comparer deux audits LLM

Reutilisez `audit_contract()` (section 4) pour auditer un contrat vulnerable avec deux
providers et comparer les rapports.

**Indice** : `audit_contract(code, client_type="anthropic")` vs `client_type="openai"`.


In [20]:
# Exercice 4 - Comparer deux audits LLM
# TODO etudiant : auditer un meme contrat vulnerable avec deux client_type et comparer
# Etape 1 : Definir contrat_vulnerable (par ex. usage dangereux de tx.origin)
# Etape 2 : Appeler audit_contract(code, client_type="anthropic") et "openai"
# Etape 3 : Comparer longueur des rapports et severites citees
contrat_vulnerable = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract PhishableWallet {
    address public owner;
    constructor() { owner = msg.sender; }

    function transferTo(address payable dest, uint256 amount) external {
        require(tx.origin == owner, "Not owner");
        dest.transfer(amount);
    }
}
"""
# rapport_anthropic = audit_contract(contrat_vulnerable, client_type="anthropic")
# rapport_openai = audit_contract(contrat_vulnerable, client_type="openai")
# print("Longueur anthropic :", len(rapport_anthropic))
# print("CRITICAL (anthropic) :", rapport_anthropic.count("CRITICAL"))
print("Exercice a completer")


Exercice a completer
